## Read the data

In [15]:
import numpy as np
import pandas as pd
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)
METHOD = "hmm_pomegranate"

In [16]:
DATA_PATH = 'D:/Google Drive/Work/ITMO/data/length_dependent_hmm/per_length_per_kfold_split/' # windows
# DATA_PATH = '/home/denklewer/Data/length_dependent_hmm/' #ubuntu

RUNS = 30
TOTAL_SPLITS = 1

# Read models

In [17]:
MODELS_DESCRIPTIONS = {
           "class_I_simple_clustering": {
        "path": "experiment_results/paper2_core_clustering/sorted_models/",
        "states": 15,
        "params_string": "60-min_iters-4-aaas",
        "combination_size": 2,
        "alleles": [
                # 'HLA-DRB1*01:01'
                    'HLA-A*01:01',
                    'HLA-A*03:01'
                    ],
        "target_lengths": [8, 9,10, 11, 12,13, 14],
        "core_start_state": [15,16],
        "core_end_state": [17,18],
        "need_to_delete": [],
        "skip_states": [],

    },
               "class_I_simple_clustering_simulated": {
        "path": "experiment_results/paper2_core_clustering/sorted_models/",
        "states": 15,
        "params_string": "60-min_iters-4-aaas",
        "combination_size":1,
        "alleles": [
             "dummy_allele"
                # 'HLA-DRB1*01:01'

                    ],
        "target_lengths": [ 9,10, 11],
        "core_start_state": [15,16],
        "core_end_state": [17,18],
        "need_to_delete": [],
        "skip_states": [],

    },




    # "mix_of_alleles_balanced":
    # {
    #     "alleles": ['HLA-A*01:01', 'HLA-A*03:01', 'HLA-A*02:01'],
    #     "path": "experiment_results/alleles_with_groups_balanced/",
    #     "states": 25,
    #     "params_string": "5-min_iters",
    #     "target_lengths": [9, 10]
    # }
}

In [18]:
selected_models = [
     "class_I_simple_clustering_simulated"
]
models_to_read = ["v"]

In [19]:
from hmm_logic_methods import load_model

In [20]:
import itertools

name_to_run = dict()
name_to_split = dict()
name_to_type = dict()
name_to_model_root = dict()
name_to_target_lengths = dict()
selected_binding_models = list()
per_name_models = dict()


def populate_models_infos(model_name, run_num, split_num, model_path, model_type, target_lengths ):
    selected_binding_models.append(model_name)

    name_to_run[model_name] = run_num
    name_to_split[model_name] = split_num
    name_to_model_root[model_name] = model_path
    assert model_type in ["bw", "v", "bwv", "bw-it", "bwv-it", "v-it"] or "mixed" in model_type
    name_to_type[model_name] = model_type
    name_to_target_lengths[model_name] = target_lengths

for model in selected_models:
    model_description = MODELS_DESCRIPTIONS[model]
    combination_size = model_description['combination_size']
    model_alleles = model_description['alleles']
    MODELS_PATH = model_description['path']
    params_string = model_description['params_string']
    states = model_description['states']
    TARGET_LENGTHS = model_description['target_lengths']
    target_length_string = "-".join([str(l) for l in TARGET_LENGTHS])
    MIXED_ALLELES = list()

    for combination in itertools.combinations(sorted(model_alleles), combination_size):
        allele_name = "+".join(combination)
        # allele_name = f"{allele_name1}+{allele_name2}"
        MIXED_ALLELES.append(allele_name)

    for allele_name in MIXED_ALLELES:
        target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_').replace('/', '_')

        for run_num_b in range(RUNS):
            for split_num_b in range(TOTAL_SPLITS):
                #bw
                if 'bw' in models_to_read:
                    bw_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-baum-welch-alg-{params_string}_model-{target_allele_name}-{split_num_b}"
                    print(bw_model_name)
                    populate_models_infos(bw_model_name, run_num_b,split_num_b, MODELS_PATH, f"bw-it-mixed({target_allele_name})", TARGET_LENGTHS)
                if 'bwv' in models_to_read:
                    #bwv
                    bwv_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-baum-welch-alg-{params_string}_model-{target_allele_name}-{split_num_b}_plus_viterbi"
                    populate_models_infos(bwv_model_name, run_num_b,split_num_b, MODELS_PATH, f"bwv-it-mixed({target_allele_name})", TARGET_LENGTHS)
                if 'v' in models_to_read:
                    #v
                    v_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-viterbi-alg-{params_string}_model-{target_allele_name}-{split_num_b}"
                    populate_models_infos(v_model_name, run_num_b,split_num_b, MODELS_PATH, f"v-it-mixed({target_allele_name})", TARGET_LENGTHS)



for model_name in selected_binding_models:
    per_name_models[model_name] =  load_model(name_to_model_root[model_name], model_name)

In [21]:
per_name_models.keys()

dict_keys(['0_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0', '1_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0', '2_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0', '3_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0', '4_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0', '5_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0', '6_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0', '7_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0', '8_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0', '9_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0', '10_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0', '11_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0', '12_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0'

## Reconstruct models

In [22]:
name_to_model_root

{'0_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0': 'experiment_results/paper2_core_clustering/sorted_models/',
 '1_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0': 'experiment_results/paper2_core_clustering/sorted_models/',
 '2_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0': 'experiment_results/paper2_core_clustering/sorted_models/',
 '3_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0': 'experiment_results/paper2_core_clustering/sorted_models/',
 '4_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0': 'experiment_results/paper2_core_clustering/sorted_models/',
 '5_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0': 'experiment_results/paper2_core_clustering/sorted_models/',
 '6_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0': 'experiment_results/paper2_core_clustering/sorted_models/',
 '7_run-9-10-11-15-st-viterbi-alg-60-min_

In [23]:
for model_name, model in per_name_models.items():
    states_to_delete = MODELS_DESCRIPTIONS[selected_models[0]]['need_to_delete']
    actual_states = [state for i, state in enumerate(model.states) if  i in states_to_delete]
    for actual_state in actual_states:
        model.graph.remove_node(actual_state)
    model.bake(verbose=1)

## Save models in same directories but in subdirectory of reconstructed

In [24]:
from hmm_visualization_methods import *

In [25]:
from hmm_logic_methods import save_model

per_name_target_path_to_results = dict()
for i, (model_name, model) in enumerate(per_name_models.items()):
    TARGET_PATH_TO_RESULTS = f"{'/'.join(name_to_model_root[model_name].split('/')[:-2])}/reconstructed_models/"
    per_name_target_path_to_results[model_name] = TARGET_PATH_TO_RESULTS
    save_model(TARGET_PATH_TO_RESULTS, model,None)


In [26]:
%%time
import os

for i, model in enumerate(per_name_models.values()):
    TARGET_PATH_TO_RESULTS = per_name_target_path_to_results[model.name]
    print(i)
    # with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph.png", 'w+') as f:
    #     model.plot(file=f, crop_zero=False)
    with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph_cropped.png", 'w+') as f:
        model.plot(file=f, crop_zero=True)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
CPU times: total: 219 ms
Wall time: 5.72 s


In [27]:
%%time

for i, model in enumerate(per_name_models.values()):
    TARGET_PATH_TO_RESULTS = per_name_target_path_to_results[model.name]
    plot_distributions_for_states(model,TARGET_PATH_TO_RESULTS, horizontal=False, discrete=True, initial_params=None)
    print(i)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
CPU times: total: 1min 20s
Wall time: 1min 20s


In [28]:
from hmm_visualization_methods import *


per_name_pyviz = dict()
for name, model in per_name_models.items():
    TARGET_PATH_TO_RESULTS = per_name_target_path_to_results[model.name]
    per_name_pyviz[name] = make_pyviz_graph(model, TARGET_PATH_TO_RESULTS, precision=3)

Converting graph for model (0_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0) (Delete zero edges)
Converting graph for model (1_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0) (Delete zero edges)
Converting graph for model (2_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0) (Delete zero edges)
Converting graph for model (3_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0) (Delete zero edges)
Converting graph for model (4_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0) (Delete zero edges)
Converting graph for model (5_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0) (Delete zero edges)
Converting graph for model (6_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0) (Delete zero edges)
Converting graph for model (7_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0) (Delete zero edges)
Converting graph for mod

In [29]:
name_to_model_root[model_name]

'experiment_results/paper2_core_clustering/sorted_models/'